# AutoImputation

This notebook demonstrates the functionality of the `autoimpute` module, which provides an automated approach to selecting and applying optimal imputation methods for missing data. Rather than manually testing different approaches, `autoimpute` evaluates multiple methods (tuning their hyperparameters to the specific dataset), identifies which performs best for your specific data, and applies it to generate high-quality imputations.

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.datasets import load_diabetes
import warnings

# Set pandas display options to limit table width
pd.set_option("display.width", 600)
pd.set_option("display.max_columns", 10)
pd.set_option("display.expand_frame_repr", False)

from microimpute.comparisons.autoimpute import autoimpute
from microimpute.visualizations.plotting import method_comparison_results

Error importing in API mode: ImportError("dlopen(/Users/movil1/envs/pe3.13/lib/python3.13/site-packages/_rinterface_cffi_api.abi3.so, 0x0002): Library not loaded: /Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib\n  Referenced from: <668E1903-F0E7-30D5-BA27-15F8287F87F7> /Users/movil1/envs/pe3.13/lib/python3.13/site-packages/_rinterface_cffi_api.abi3.so\n  Reason: tried: '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file)")
Trying to import in ABI mode.
/Users/movil1/envs/pe3.13/lib/python3.13/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "PWD" redefined by R and overriding existing variable. Current: "/", R: "/Users/movil1/Desktop/PYTHONJOBS/Policy

## Data preparation

This demonstration uses the diabetes dataset from scikit-learn. In real-world imputation scenarios, you would typically have a "donor" dataset with complete information for both predictor and target variables, and a "receiver" dataset that lacks some target variables that need to be imputed.

In [2]:
# Load the diabetes dataset
diabetes = load_diabetes()
diabetes_data = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)

# Display the first few rows to understand the data structure
diabetes_data.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641


For this demonstration, the diabetes dataset is split into donor and receiver portions. Part of the data is treated as the donor dataset with complete information, and another part as the receiver dataset with some variables that need imputation. `autoimpute` handles imputation of numerical, categorical and boolean variables, lifting constraints on the choice of data sets and variables.

In [3]:
# Split the data into donor and receiver portions
donor_indices = np.random.choice(
    len(diabetes_data), size=int(0.7 * len(diabetes_data)), replace=False
)
receiver_indices = np.array(
    [i for i in range(len(diabetes_data)) if i not in donor_indices]
)

donor_data = diabetes_data.iloc[donor_indices].reset_index(drop=True)
receiver_data = diabetes_data.iloc[receiver_indices].reset_index(drop=True)

# Define which variables we'll use as predictors and which we want to impute
predictors = ["age", "sex", "bmi", "bp"]
imputed_variables = ["s1", "s4"]

# For demonstration purposes, we'll remove the variables we want to impute from the receiver dataset
receiver_data_without_targets = receiver_data.drop(columns=imputed_variables)

print(f"Donor data shape: {donor_data.shape}")
print(f"Receiver data shape: {receiver_data_without_targets.shape}")
print(f"Predictors: {predictors}")
print(f"Variables to impute: {imputed_variables}")

Donor data shape: (309, 10)
Receiver data shape: (133, 8)
Predictors: ['age', 'sex', 'bmi', 'bp']
Variables to impute: ['s1', 's4']


## Running `autoimpute` 

Use the `autoimpute` function to automatically evaluate different imputation methods, select the best one, and generate imputations. The function handles all the complexity of model evaluation, selection, and application in a single call.

In [4]:
warnings.filterwarnings("ignore")

# Run the autoimpute process
results = autoimpute(
    donor_data=donor_data,
    receiver_data=receiver_data_without_targets,
    predictors=predictors,
    imputed_variables=imputed_variables,
    tune_hyperparameters=False,  # enable automated hyperparameter tuning if desired
    k_folds=3,  # Number of cross-validation folds
)

print(
    f"Shape of receiver data before imputation: {receiver_data_without_targets.shape} \nShape of receiver data after imputation: {results.receiver_data.shape}"
)

Evaluating models:   0%|          | 0/4 [00:00<?, ?it/s]

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   3 out of   3 | elapsed:    3.4s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Batch computation too fast (0.1890571117401123s.) Setting batch_size=2.
[Parallel(n_jobs=-1)]: Done   3 out of   3 | elapsed:    0.2s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   3 out of   3 | elapsed:    0.9s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   3 out of   3 | elapsed:    3.6s finished


Shape of receiver data before imputation: (133, 8) 
Shape of receiver data after imputation: (133, 10)


## Understanding the results

The `autoimpute` function returns four key objects that provide comprehensive information about the imputation process:

- `imputations`: A dictionary where keys are quantiles used for imputation and values are DataFrames containing the imputed values at each quantile
- `imputed_data`: The receiver dataset with imputed values integrated into it
- `fitted_model`: The best-performing imputation model, already fitted on the donor data
- `method_results_df`: A DataFrame with detailed performance metrics for all evaluated imputation methods

In [5]:
# Examine the comparative performance of different imputation methods
print("Cross-validation results for different imputation methods:")
results.cv_results

Cross-validation results for different imputation methods:


,0.05,0.1,0.15,0.2,0.25,...,0.8,0.85,0.9,0.95,mean_loss
QRF,0.005206,0.007714,0.010652,0.013726,0.017578,...,0.016469,0.013081,0.010507,0.006769,0.016371
OLS,0.003969,0.006727,0.009112,0.011131,0.012739,...,0.013079,0.011218,0.008765,0.005431,0.012667
QuantReg,0.003882,0.006577,0.009084,0.011261,0.012793,...,0.013183,0.011169,0.008899,0.005258,0.012713
Matching,0.024895,0.024740,0.024585,0.024430,0.024275,...,0.022570,0.022415,0.022260,0.022105,0.023500


The table above provides a comprehensive view of how each imputation method performs across different quantiles. The 'mean_loss' column shows the average quantile loss across all quantiles for each method. Lower values indicate better performance, and `autoimpute` automatically selects the method with the lowest average loss.

In [6]:
# Identify which method was selected as the best performer
best_method = results.cv_results["mean_loss"].idxmin()
print(f"Best performing method: {best_method}")
print(f"Average loss: {results.cv_results.loc[best_method, 'mean_loss']:.4f}")

Best performing method: OLS
Average loss: 0.0127


## Visualizing method comparison

Visualize how different methods perform across quantiles provides insight into which methods are most appropriate for different parts of the distribution.

In [7]:
# Extract the quantiles used in the evaluation
quantiles = [q for q in results.cv_results.columns if isinstance(q, float)]

comparison_viz = method_comparison_results(
    data=results.cv_results,
    metric_name="Test Quantile Loss",
    data_format="wide",
)
fig = comparison_viz.plot(
    title="Autoimpute Method Comparison",
    show_mean=True,
)
fig.show()

The plot above illustrates how each imputation method performs across different quantiles of the distribution. Methods with consistently lower lines generally perform better overall.

In [8]:
comparison_viz.summary()

,Method,Mean Test Quantile Loss,Best Quantile,Best Test Quantile Loss,Worst Quantile,Worst Test Quantile Loss
1,OLS,0.012667,0.05,0.003969,0.55,0.016917
2,QuantReg,0.012713,0.05,0.003882,0.55,0.017056
0,QRF,0.016371,0.05,0.005206,0.40,0.022553
3,Matching,0.023500,0.95,0.022105,0.05,0.024895


By calling `summary` on the object returned by `method_comparison_results` function, you can get a summary of the imputation results, including the mean and standard deviation of the quantile loss for each method. This summary can help you understand the performance of different imputation methods in a more concise manner.

## Examining the imputed values

Now let us assess the actual imputed values generated by the best-performing method.

In [9]:
# Examine imputed values (these were imputed for q=0.5 by default)
median_imputations = results.imputations[
    "best_method"
]  # Extract the best imputations with the "best_method" key
print("Median imputed values:")
median_imputations.head()

Median imputed values:


,s1,s4
0,0.015336,0.038018
1,0.019831,0.036004
2,-0.020689,-0.005872
3,0.015436,0.021340
4,-0.029310,-0.050130


In [10]:
# Look at the full receiver dataset with imputed values integrated
print("Receiver dataset with imputed values:")
results.receiver_data.head()

Receiver dataset with imputed values:


,age,sex,bmi,bp,s2,s3,s5,s6,s1,s4
0,0.038076,0.050680,0.061696,0.021872,-0.034821,-0.043401,0.019907,-0.017646,0.015336,0.038018
1,0.085299,0.050680,0.044451,-0.005670,-0.034194,-0.032356,0.002861,-0.025930,0.019831,0.036004
2,-0.045472,0.050680,-0.047163,-0.015999,-0.024800,0.000779,-0.062917,-0.038357,-0.020689,-0.005872
3,0.063504,0.050680,-0.001895,0.066629,0.108914,0.022869,-0.035816,0.003064,0.015436,0.021340
4,-0.096328,-0.044642,-0.083808,0.008101,-0.090561,-0.013948,-0.062917,-0.034215,-0.029310,-0.050130


## Evaluating imputation quality

In this demonstration, since the receiver dataset was artificially created by removing variables from the original data, there exists the unique opportunity to evaluate the quality of our imputations by comparing them to the actual values.

In [11]:
# Visualize comparison between actual and imputed values
for var in imputed_variables:
    fig = go.Figure()

    # Plot actual values
    fig.add_trace(
        go.Scatter(
            x=receiver_data.index,
            y=receiver_data[var],
            mode="markers",
            name="Actual values",
            marker=dict(color="blue", size=8),
        )
    )

    # Plot imputed values
    fig.add_trace(
        go.Scatter(
            x=results.receiver_data.index,
            y=results.receiver_data[var],
            mode="markers",
            name="Imputed values",
            marker=dict(color="red", size=8),
        )
    )

    # Customize the plot appearance
    fig.update_layout(
        title=f"Comparison of actual vs imputed values for {var}",
        xaxis_title="Sample Index",
        yaxis_title=f"{var} Value",
        legend_title="Type",
        hovermode="closest",
    )

    fig.show()

The plots above show how well the imputed values (red) match the actual values (blue) that were removed from the receiver dataset. This visual comparison helps assess the quality of the imputations generated by the best-performing method.

## Advanced usage

### Custom models and hyperparameters

The `autoimpute` function allows for customization of both the models to evaluate and their hyperparameters. This flexibility enables adaptation to specific dataset characteristics and imputation requirements. The models that support hyperparameter specification and tuning are Matching and QRF.

In [12]:
from microimpute.models import *

# Specify a custom subset of models to evaluate
custom_models = [QRF, OLS, Matching]

# Specify custom hyperparameters for some models
custom_hyperparameters = {
    "QRF": {"n_estimators": 200, "max_depth": 10},
    "Matching": {"constrained": True},
}

# Then simply run autoimpute with custom models and hyperparameters
advanced_results = autoimpute(
    donor_data=donor_data,
    receiver_data=receiver_data_without_targets,
    predictors=predictors,
    imputed_variables=imputed_variables,
    models=custom_models,
    hyperparameters=custom_hyperparameters,
    k_folds=3,
)

advanced_results.imputations["best_method"]

Evaluating models:   0%|          | 0/3 [00:00<?, ?it/s]

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   3 out of   3 | elapsed:    1.4s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Batch computation too fast (0.051928043365478516s.) Setting batch_size=2.
[Parallel(n_jobs=-1)]: Done   3 out of   3 | elapsed:    0.1s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   3 out of   3 | elapsed:    2.4s finished


,s1,s4
0,0.015336,0.038018
1,0.019831,0.036004
2,-0.020689,-0.005872
3,0.015436,0.021340
4,-0.029310,-0.050130
...,...,...
128,-0.003101,0.017096
129,-0.013886,-0.029767
130,-0.012898,0.006271
131,0.004467,0.013287


### Comparison of imputed values across models

For comparing, not only performance through quantile loss, but also final results through the evaluation of imputed values, `autoimpute` supports setting the parameter `impute_all` to True so that it will not only perform impuation with the model chosen as the best performing but also all others. When set to True, this parameter ensures that `autoimpute`'s results base clase contains an imputations dictionary and fitted models dictionary for all other models in addition to the "best_method".

In [13]:
warnings.filterwarnings("ignore")

# Run the autoimpute process
results = autoimpute(
    donor_data=donor_data,
    receiver_data=receiver_data_without_targets,
    predictors=predictors,
    imputed_variables=imputed_variables,
    tune_hyperparameters=False,
    impute_all=True,
    k_folds=3,
)

print(f"Imputation results available for models: {results.imputations.keys()}")
print(
    f"The best performing model is: {results.fitted_models['best_method'].__class__.__name__}"
)

Evaluating models:   0%|          | 0/4 [00:00<?, ?it/s]

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   3 out of   3 | elapsed:    1.0s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Batch computation too fast (0.06503176689147949s.) Setting batch_size=2.
[Parallel(n_jobs=-1)]: Done   3 out of   3 | elapsed:    0.1s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   3 out of   3 | elapsed:    0.5s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   3 out of   3 | elapsed:    2.3s finished
{0.5:            s1        s4
0   -0.034592 -0.002592
1    0.052093  0.071210
2   -0.000193 -0.002592
3    0.035582  0.034309
4    0.002559 -0.002592
..        ...       ...
128  0.061725  0.108111
129 -0.056607 -0.076395
130 -0.038720 -0.039493
131  0.006687  0.034309
132 -0.051103 -0.076395

[133 rows x 2 columns

Imputation results available for models: dict_keys(['best_method', 'QRF', 'QuantReg', 'Matching'])
The best performing model is: OLSResults
